# Model selection
- For sections 3.4 (Modelling), 3.5 (Evaluation) and for Table 4 (Model Performance - Breach Class) in the report
- Input file: `incident_event_log.csv` (the raw data)

#### Load the libraries

In [41]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score, precision_score, f1_score

#### Read the raw data

In [42]:
data = pd.read_csv("incident_event_log.csv", dtype=str, keep_default_na=False)
print("Rows and columns:", data.shape)

Rows and columns: (141712, 36)


#### Make one row per incident

In [43]:
# Sort the records of each incident from first to last
data["sys_mod_count"] = data["sys_mod_count"].astype(int)
data = data.sort_values(["number", "sys_mod_count"])

first_record = data.groupby("number").head(1).set_index("number")
last_record = data.groupby("number").tail(1).set_index("number")

# The columns that are known when a ticket is created
columns = ["priority", "impact", "urgency", "category", "subcategory", "u_symptom",
           "assignment_group", "contact_type", "location"]

tickets = first_record[columns].copy()
tickets = tickets.replace("?", "unknown")      

# Add the hour and the weekday for open tickets
tickets["opened_at"] = pd.to_datetime(first_record["opened_at"], format="%d/%m/%Y %H:%M")
tickets["opened_hour"] = tickets["opened_at"].dt.hour.astype(str)
tickets["opened_weekday"] = tickets["opened_at"].dt.day_name()

# The label: 1 = the ticket missed its SLA (breach), 0 = it met its SLA
tickets["breach"] = 0
tickets.loc[last_record["made_sla"].str.lower() == "false", "breach"] = 1

print("Incidents:", len(tickets))
print("Breach rate:", round(tickets["breach"].mean() * 100, 1), "%")

Incidents: 24918
Breach rate: 36.6 %


#### Split the tickets by time
The **oldest 80%** are for training. The **newest 20%** are for testing.

In [44]:
# Sort from oldest to newest
tickets = tickets.sort_values("opened_at", kind="mergesort")

features = columns + ["opened_hour", "opened_weekday"]

split_position = int(len(tickets) * 0.8)
cutoff = tickets["opened_at"].iloc[split_position]

train = tickets[tickets["opened_at"] < cutoff]
test = tickets[tickets["opened_at"] >= cutoff]

print("Cut-off date:", cutoff)

print("Training tickets:", len(train))
print("Test tickets:", len(test))

print("Breach rate in training:", round(train["breach"].mean() * 100, 1), "%")
print("Breach rate in test:", round(test["breach"].mean() * 100, 1), "%")

Cut-off date: 2016-05-11 00:29:00
Training tickets: 19934
Test tickets: 4984
Breach rate in training: 40.0 %
Breach rate in test: 23.0 %


#### Make a validation set
I take the newest **20%** of the training tickets as a validation set.

In [45]:
validation_position = int(len(train) * 0.8)
train_part = train.iloc[:validation_position]
validation = train.iloc[validation_position:]

print("Tickets to learn:", len(train_part))
print("Validation tickets:", len(validation))

Tickets to learn: 15947
Validation tickets: 3987


#### Two small helper functions
- The **first "encode" turns the categories into numbers (one-hot encoding)
- The second "show_scores" prints the three scores

In [46]:
# First help
def encode(learn_data, other_data):
    encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20)
    learn_encoded = encoder.fit_transform(learn_data[features])
    other_encoded = encoder.transform(other_data[features])
    return learn_encoded, other_encoded

# Second help
def show_scores(name, real, predicted):
    recall = recall_score(real, predicted)
    precision = precision_score(real, predicted, zero_division=0)
    f1 = f1_score(real, predicted)
    print(name, "| recall", round(recall, 3), "| precision", round(precision, 3), "| F1", round(f1, 3))
    return f1

#### Choose the tree depth
I try depths from 2 to 10 on the validation set.

In [47]:
X_learn, X_validation = encode(train_part, validation)

for depth in range(2, 11):
    tree = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    tree.fit(X_learn, train_part["breach"])
    predicted = tree.predict(X_validation)
    show_scores("depth " + str(depth), validation["breach"], predicted)

depth 2 | recall 0.727 | precision 0.371 | F1 0.491
depth 3 | recall 0.727 | precision 0.369 | F1 0.49
depth 4 | recall 0.727 | precision 0.369 | F1 0.49
depth 5 | recall 0.727 | precision 0.369 | F1 0.49
depth 6 | recall 0.727 | precision 0.369 | F1 0.49
depth 7 | recall 0.696 | precision 0.373 | F1 0.486
depth 8 | recall 0.659 | precision 0.391 | F1 0.491
depth 9 | recall 0.66 | precision 0.37 | F1 0.474
depth 10 | recall 0.703 | precision 0.364 | F1 0.48


**Result:** depth 2 has the highest F1 score (0.491)

#### Choose the alert threshold
The logistic regression gives each ticket a risk score between 0 and 1

In [48]:
model = LogisticRegression(class_weight="balanced", max_iter=2000)
model.fit(X_learn, train_part["breach"])
risk_scores = model.predict_proba(X_validation)[:, 1]

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    predicted = (risk_scores >= threshold).astype(int)
    flagged = round(predicted.mean() * 100, 1)
    show_scores("threshold " + str(threshold), validation["breach"], predicted)
    print("tickets flagged:", flagged, "%")

threshold 0.3 | recall 0.931 | precision 0.298 | F1 0.452
tickets flagged: 68.0 %
threshold 0.35 | recall 0.889 | precision 0.324 | F1 0.475
tickets flagged: 59.7 %
threshold 0.4 | recall 0.803 | precision 0.348 | F1 0.486
tickets flagged: 50.2 %
threshold 0.45 | recall 0.702 | precision 0.378 | F1 0.492
tickets flagged: 40.4 %
threshold 0.5 | recall 0.614 | precision 0.422 | F1 0.5
tickets flagged: 31.7 %
threshold 0.55 | recall 0.528 | precision 0.487 | F1 0.507
tickets flagged: 23.6 %
threshold 0.6 | recall 0.417 | precision 0.521 | F1 0.463
tickets flagged: 17.4 %
threshold 0.65 | recall 0.341 | precision 0.553 | F1 0.422
tickets flagged: 13.4 %
threshold 0.7 | recall 0.27 | precision 0.589 | F1 0.37
tickets flagged: 10.0 %


**Result:** 
- 0.55 has the highest F1 score, but it finds only 53% of the breaches
- 0.50 has almost the same F1 score and finds 61%.
- A threshold of 0.40 flags half of all tickets, which is too many. I choose **0.50**.

#### Final test (Table 4)

In [50]:
X_train, X_test = encode(train, test)
real = test["breach"]

# 1) Baseline: a ticket with priority 1 or 2 is predicted to breach
baseline = test["priority"].isin(["1 - Critical", "2 - High"]).astype(int)
show_scores("Baseline (priority 1 or 2)", real, baseline)

# 2) Logistic regression with the threshold 0.50
model = LogisticRegression(class_weight="balanced", max_iter=2000)
model.fit(X_train, train["breach"])
predicted = (model.predict_proba(X_test)[:, 1] >= 0.50).astype(int)
show_scores("Logistic regression", real, predicted)

# 3) Decision tree with "depth 2"
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X_train, train["breach"])
predicted = tree.predict(X_test)
show_scores("Decision tree (depth 2)", real, predicted)

# For comparison: always saying "no breach"
print("Accuracy ticket with 'no breach':", round(1 - real.mean(), 3))

Baseline (priority 1 or 2) | recall 0.075 | precision 0.896 | F1 0.138
Logistic regression | recall 0.695 | precision 0.429 | F1 0.53
Decision tree (depth 2) | recall 0.784 | precision 0.387 | F1 0.518
Accuracy ticket with 'no breach': 0.77
